# Lab 5: Frameworks and automatic differentiation

Run from the `lab05/` folder: `jupyter nbconvert --execute --to notebook --inplace lab5.ipynb` or `python src/lab5.py`. Code lives in `src/`.

| Task | Where |
|---|---|
| 1. `Value` node (data, grad, parents, local derivative rule) | `src/engine.py` → class `Value` |
| 2. add, mul, pow, ReLU build the graph in the forward pass | `src/engine.py` → `__add__`, `__mul__`, `__pow__`, `relu` |
| 3. backward: topological sort, seed grad = 1, reverse order, `+=` accumulation | `src/engine.py` → `Value.backward` + accumulation test |
| 4. check vs PyTorch autograd and numerical gradients, max abs diff < 1e-6 | Task 4 cell |
| 5. make_moons, 2→8→1, sigmoid, BCE, speed vs PyTorch, median of 5 runs | Task 5 cell |
| 6. graph inspection, eager vs `torch.compile` / `torch.jit.trace`, shape/dtype/contiguity, non-contiguous cost | Task 6 cell |


In [ ]:
import sys, os, importlib, platform
sys.path.insert(0, 'src')
import torch
import lab5; importlib.reload(lab5)
lab5.RESULTS['env']={'python':platform.python_version(),'torch':torch.__version__,'platform':platform.platform(),'cpu_count':os.cpu_count()}
print(lab5.RESULTS['env'])

## Tasks 1–3: engine + gradient accumulation test
`x*x + x` uses `x` three times, so the correct gradient (7) is only obtained if gradients are **accumulated** (`+=`), not overwritten.

In [ ]:
lab5.task3_accumulation_test()

# small graph printout: every op created a node with parents
from engine import Value
a, b, c = Value(2.0, label='a'), Value(-3.0, label='b'), Value(1.0, label='c')
out = ((a*b + c)**2).relu()
topo = out.backward()
for n in topo:
    print(f"{n._op or 'leaf':8} data={n.data:8.3f} grad={n.grad:8.3f} parents={len(n._prev)}")

## Task 4: `((a*b + c)**2).relu().sum()` — our engine vs PyTorch autograd vs central differences (h = 1e-6), float64

In [ ]:
lab5.task4()

## Task 5: make_moons (100 samples), 2 → 8 (ReLU) → 1 (sigmoid), BCE, full-batch SGD lr = 0.5, 100 epochs
Both models start from **identical weights**, so their loss curves must coincide. Timing = median of 5 full training runs (PyTorch limited to 1 thread for fairness). The pure-Python run takes about 1 minute in total.

In [ ]:
lab5.task5()
from IPython.display import Image, display
display(Image('results/task5_moons.png'))

## Task 6: PyTorch graph, eager vs compiled, tensor layout

In [ ]:
lab5.task6()

## Summary of graded evidence

In [ ]:
import json
R = lab5.RESULTS
print('Task 4: max abs diff =', f"{R['task4']['max_abs_diff']:.2e}", '->', 'PASS' if R['task4']['pass'] else 'FAIL')
t5 = R['task5']
print(f"Task 5: our engine {t5['median_ms_ours']:.0f} ms vs PyTorch {t5['median_ms_torch']:.1f} ms (x{t5['speedup']:.0f}), "
      f"acc {t5['acc'][0]:.2f}/{t5['acc'][1]:.2f}, max loss diff {t5['max_loss_diff']:.1e}")
t6 = R['task6']
print('Task 6:', t6['compile_mode'], f"(compile time {t6['compile_seconds']:.1f} s)")
for r in t6['eager_vs_compiled']:
    print(f"   batch {r['batch']:5d}: eager {r['eager_ms']:.4f} ms | compiled {r['compiled_ms']:.4f} ms | x{r['speedup']:.2f}")
for r in t6['noncontig']:
    print(f"   {r['op']:32} contiguous {r['contig_ms']:.2f} ms | non-contiguous {r['noncontig_ms']:.2f} ms | x{r['ratio']:.2f}")
json.dump(R, open('results/results.json','w'), indent=2, default=float)

## Short note: when does compilation help?

*(Check that the sentences match the numbers printed above before submitting.)*

**What compilation does.** In eager mode PyTorch runs each operation as soon as Python reaches it: every op is a separate Python call, dispatcher lookup and kernel launch, and every intermediate tensor is written to and read back from memory. `torch.compile` (TorchDynamo + Inductor) captures the whole forward as a graph, then fuses chains of element-wise ops (here GELU → `h*sigmoid(h)` → `+x` → LayerNorm) into a few generated kernels and removes Python overhead. `torch.jit.trace` records one execution into a static TorchScript graph, which removes Python overhead and enables some fusion.

**When it helps**
- Models that run many times with the **same shapes** (inference serving, long training loops), so the one-off compile cost (seconds, see above) is amortised.
- Models with **many small / element-wise ops** that can be fused; memory-bound work benefits most.
- **Medium-to-large batches**, where fused kernels save real memory traffic.

**When it does not help (or hurts)**
- Short scripts, notebooks and debugging: compile time dominates; eager is easier to debug.
- Workloads dominated by **large matmuls** (already in optimised BLAS) — little left to optimise.
- **Changing shapes** or data-dependent Python control flow → graph breaks / recompilation; `jit.trace` even silently freezes one control-flow path.
- Very small inputs on CPU may show little or no gain because fixed overheads remain.

**Our autograd engine vs PyTorch.** Our engine is correct (gradients match to < 1e-6) but is much slower, because every scalar operation is a Python object and a Python closure; PyTorch works on whole tensors with vectorised C++ kernels, which is the main framework overhead lesson of Chapter 7.

**Non-contiguous tensors.** `.t()` and `[:, ::2]` are *views*: same storage, different strides, so `is_contiguous()` is False. Ops on them read memory with jumps (poor cache use / no straight vectorisation), so they are slower than on contiguous tensors; calling `.contiguous()` costs one copy but makes repeated later ops fast.
